In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

# =========================================================
# CONFIGURATION
# =========================================================

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"

NS = f"{CATALOG}.{SCHEMA}"

spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

print("Namespace:", NS)

# =========================================================
# LOAD BRONZE TABLES
# =========================================================

bronze_products = spark.table(
    f"{NS}.bronze_product_limits"
)

bronze_shipments = spark.table(
    f"{NS}.bronze_shipments"
)

bronze_deliveries = spark.table(
    f"{NS}.bronze_delivery_events"
)

# =========================================================
# COUNTS
# =========================================================

product_count = bronze_products.count()
shipment_count = bronze_shipments.count()
delivery_count = bronze_deliveries.count()

print("")
print("======================================")
print("REFERENCE BRONZE COUNTS")
print("======================================")

print("Products   :", product_count)
print("Shipments  :", shipment_count)
print("Deliveries :", delivery_count)

print("======================================")


In [0]:
from pyspark.sql import functions as F

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"
NS = f"{CATALOG}.{SCHEMA}"

# =========================================================
# LOAD BRONZE PRODUCTS
# =========================================================

bronze_products = spark.table(
    f"{NS}.bronze_product_limits"
)

# =========================================================
# APPLY VALIDATION RULES
# =========================================================

validated_products = (
    bronze_products
    .withColumn(
        "rejection_reason",

        F.when(
            F.col("product_id").isNull(),
            F.lit("missing_product_id")
        )

        .when(
            F.col("min_temperature").isNull(),
            F.lit("missing_min_temperature")
        )

        .when(
            F.col("max_temperature").isNull(),
            F.lit("missing_max_temperature")
        )

        .when(
            F.col("min_temperature")
            > F.col("max_temperature"),
            F.lit("invalid_temperature_range")
        )

        .otherwise(
            F.lit(None).cast("string")
        )
    )
)

# =========================================================
# DISPLAY VALIDATION
# =========================================================

print("===== PRODUCT VALIDATION =====")

display(
    validated_products.select(
        "product_id",
        "min_temperature",
        "max_temperature",
        "rejection_reason"
    )
)

In [0]:
from pyspark.sql import functions as F

# =========================================================
# PRODUCT SILVER + QUARANTINE
# Complete standalone cell
# =========================================================

# ---------------------------------------------------------
# 1. CONFIGURATION
# ---------------------------------------------------------

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"

NS = f"{CATALOG}.{SCHEMA}"

print("Namespace:", NS)


# ---------------------------------------------------------
# 2. LOAD BRONZE PRODUCT TABLE
# ---------------------------------------------------------

bronze_products = spark.table(
    f"{NS}.bronze_product_limits"
)

print("")
print("===== BRONZE PRODUCTS =====")

display(bronze_products)

bronze_product_count = bronze_products.count()

print(
    "Bronze product count:",
    bronze_product_count
)


# ---------------------------------------------------------
# 3. APPLY PRODUCT VALIDATION RULES
# ---------------------------------------------------------

validated_products = (
    bronze_products
    .withColumn(
        "rejection_reason",

        F.when(
            F.col("product_id").isNull(),
            F.lit("missing_product_id")
        )

        .when(
            F.col("min_temperature").isNull(),
            F.lit("missing_min_temperature")
        )

        .when(
            F.col("max_temperature").isNull(),
            F.lit("missing_max_temperature")
        )

        .when(
            F.col("min_temperature")
            > F.col("max_temperature"),
            F.lit("invalid_temperature_range")
        )

        .otherwise(
            F.lit(None).cast("string")
        )
    )
)


# ---------------------------------------------------------
# 4. DISPLAY VALIDATION RESULTS
# ---------------------------------------------------------

print("")
print("===== PRODUCT VALIDATION RESULTS =====")

display(
    validated_products.select(
        "product_id",
        "min_temperature",
        "max_temperature",
        "rejection_reason"
    )
)


# ---------------------------------------------------------
# 5. CREATE VALID PRODUCTS DATAFRAME
# ---------------------------------------------------------

valid_products = (
    validated_products
    .filter(
        F.col("rejection_reason").isNull()
    )
)


# ---------------------------------------------------------
# 6. CREATE QUARANTINE PRODUCTS DATAFRAME
# ---------------------------------------------------------

quarantine_products = (
    validated_products
    .filter(
        F.col("rejection_reason").isNotNull()
    )
)


# ---------------------------------------------------------
# 7. COUNT VALID / INVALID PRODUCTS
# ---------------------------------------------------------

valid_product_count = valid_products.count()

quarantine_product_count = quarantine_products.count()

print("")
print("===== PRODUCT COUNTS =====")

print(
    "Bronze products:",
    bronze_product_count
)

print(
    "Valid products:",
    valid_product_count
)

print(
    "Quarantined products:",
    quarantine_product_count
)


# ---------------------------------------------------------
# 8. RECONCILIATION
# ---------------------------------------------------------

assert (
    valid_product_count
    + quarantine_product_count
    == bronze_product_count
), (
    "FAILED: Product reconciliation failed. "
    f"Bronze={bronze_product_count}, "
    f"Valid={valid_product_count}, "
    f"Quarantine={quarantine_product_count}"
)

print("")
print("✅ Product reconciliation passed")


# ---------------------------------------------------------
# 9. PREPARE FINAL SILVER PRODUCT TABLE
# ---------------------------------------------------------

silver_product_limits = (
    valid_products
    .select(
        "product_id",
        "min_temperature",
        "max_temperature",
        "_source_file",
        "_processing_time",
        "_run_id"
    )
)


# ---------------------------------------------------------
# 10. WRITE SILVER PRODUCT TABLE
# ---------------------------------------------------------

(
    silver_product_limits.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{NS}.silver_product_limits"
    )
)

print("")
print("✅ silver_product_limits written successfully")


# ---------------------------------------------------------
# 11. PREPARE PRODUCT QUARANTINE TABLE
# ---------------------------------------------------------

quarantine_product_limits = (
    quarantine_products
    .select(
        "product_id",
        "min_temperature",
        "max_temperature",
        "rejection_reason",
        "_source_file",
        "_processing_time",
        "_run_id"
    )
)


# ---------------------------------------------------------
# 12. WRITE PRODUCT QUARANTINE TABLE
# ---------------------------------------------------------

(
    quarantine_product_limits.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{NS}.quarantine_product_limits"
    )
)

print("")
print("✅ quarantine_product_limits written successfully")


# ---------------------------------------------------------
# 13. READ SAVED TABLES BACK
# ---------------------------------------------------------

saved_silver_products = spark.table(
    f"{NS}.silver_product_limits"
)

saved_quarantine_products = spark.table(
    f"{NS}.quarantine_product_limits"
)


# ---------------------------------------------------------
# 14. DISPLAY SAVED SILVER PRODUCTS
# ---------------------------------------------------------

print("")
print("========================================")
print("         SILVER PRODUCT LIMITS")
print("========================================")

display(
    saved_silver_products
    .orderBy("product_id")
)


# ---------------------------------------------------------
# 15. DISPLAY SAVED QUARANTINE PRODUCTS
# ---------------------------------------------------------

print("")
print("========================================")
print("       QUARANTINED PRODUCT LIMITS")
print("========================================")

display(
    saved_quarantine_products
    .orderBy("product_id")
)


# ---------------------------------------------------------
# 16. FINAL SAVED TABLE COUNTS
# ---------------------------------------------------------

saved_silver_count = saved_silver_products.count()

saved_quarantine_count = saved_quarantine_products.count()

print("")
print("===== FINAL PRODUCT SILVER CHECK =====")

print(
    "Bronze products:",
    bronze_product_count
)

print(
    "Silver products:",
    saved_silver_count
)

print(
    "Quarantined products:",
    saved_quarantine_count
)


# ---------------------------------------------------------
# 17. EXPECTED FIXTURE ASSERTIONS
# ---------------------------------------------------------

assert bronze_product_count == 3, (
    f"FAILED: Expected 3 Bronze products, "
    f"found {bronze_product_count}"
)

assert saved_silver_count == 3, (
    f"FAILED: Expected 3 Silver products, "
    f"found {saved_silver_count}"
)



In [0]:
from pyspark.sql import functions as F

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"
NS = f"{CATALOG}.{SCHEMA}"

# =========================================================
# LOAD TABLES
# =========================================================

bronze_shipments = spark.table(
    f"{NS}.bronze_shipments"
)

silver_products = spark.table(
    f"{NS}.silver_product_limits"
)

# =========================================================
# CREATE VALID PRODUCT LOOKUP
# =========================================================

valid_product_ids = (
    silver_products
    .select(
        F.col("product_id")
        .alias("valid_product_id")
    )
    .distinct()
)

# =========================================================
# JOIN SHIPMENT WITH PRODUCT LOOKUP
# =========================================================

shipments_checked = (
    bronze_shipments.alias("s")
    .join(
        valid_product_ids.alias("p"),
        F.col("s.product_id")
        == F.col("p.valid_product_id"),
        "left"
    )
    .select(
        "s.*",
        F.col("p.valid_product_id")
    )
)

# =========================================================
# APPLY VALIDATION RULES
# =========================================================

validated_shipments = (
    shipments_checked
    .withColumn(
        "rejection_reason",

        F.when(
            F.col("shipment_id").isNull(),
            F.lit("missing_shipment_id")
        )

        .when(
            F.col("product_id").isNull(),
            F.lit("missing_product_id")
        )

        .when(
            F.col("valid_product_id").isNull(),
            F.lit("unknown_product")
        )

        .when(
            F.col("dispatch_time").isNull(),
            F.lit("invalid_dispatch_time")
        )

        .when(
            F.col("expected_arrival").isNull(),
            F.lit("invalid_expected_arrival")
        )

        .when(
            F.col("expected_arrival")
            < F.col("dispatch_time"),
            F.lit("invalid_shipment_time")
        )

        .otherwise(
            F.lit(None).cast("string")
        )
    )
)

# =========================================================
# DISPLAY
# =========================================================

display(
    validated_shipments.select(
        "shipment_id",
        "product_id",
        "vehicle_id",
        "origin",
        "destination",
        "dispatch_time",
        "expected_arrival",
        "rejection_reason"
    )
)

In [0]:
from pyspark.sql import functions as F

# =========================================================
# SPLIT
# =========================================================

valid_shipments = (
    validated_shipments
    .filter(
        F.col("rejection_reason").isNull()
    )
)

quarantine_shipments = (
    validated_shipments
    .filter(
        F.col("rejection_reason").isNotNull()
    )
)

# =========================================================
# COUNTS
# =========================================================

bronze_shipment_count = bronze_shipments.count()
valid_shipment_count = valid_shipments.count()
quarantine_shipment_count = quarantine_shipments.count()

print("Bronze shipments     :", bronze_shipment_count)
print("Valid shipments      :", valid_shipment_count)
print("Quarantine shipments :", quarantine_shipment_count)

# =========================================================
# RECONCILIATION
# =========================================================

assert (
    valid_shipment_count
    + quarantine_shipment_count
    == bronze_shipment_count
), "FAILED: Shipment reconciliation failed"

print("✅ Shipment reconciliation passed")

In [0]:
from pyspark.sql import functions as F

# =========================================================
# CLEAN SILVER
# =========================================================

silver_shipments = (
    valid_shipments
    .select(
        "shipment_id",
        "product_id",
        "vehicle_id",
        "origin",
        "destination",
        "dispatch_time",
        "expected_arrival",
        "_source_file",
        "_processing_time",
        "_run_id"
    )
)

# =========================================================
# WRITE SILVER
# =========================================================

(
    silver_shipments.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{NS}.silver_shipments"
    )
)

print("✅ silver_shipments written")

# =========================================================
# QUARANTINE
# =========================================================

quarantine_shipments_final = (
    quarantine_shipments
    .select(
        "shipment_id",
        "product_id",
        "vehicle_id",
        "origin",
        "destination",
        "dispatch_time",
        "expected_arrival",
        "rejection_reason",
        "_source_file",
        "_processing_time",
        "_run_id"
    )
)

(
    quarantine_shipments_final.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{NS}.quarantine_shipments"
    )
)

print("✅ quarantine_shipments written")

# =========================================================
# DISPLAY
# =========================================================

print("")
print("SILVER SHIPMENTS")

display(
    spark.table(
        f"{NS}.silver_shipments"
    )
)

In [0]:
from pyspark.sql import functions as F

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"
NS = f"{CATALOG}.{SCHEMA}"

# =========================================================
# LOAD TABLES
# =========================================================

bronze_deliveries = spark.table(
    f"{NS}.bronze_delivery_events"
)

silver_shipments_table = spark.table(
    f"{NS}.silver_shipments"
)

# =========================================================
# VALID SHIPMENT LOOKUP
# =========================================================

valid_shipment_ids = (
    silver_shipments_table
    .select(
        F.col("shipment_id")
        .alias("valid_shipment_id")
    )
    .distinct()
)

# =========================================================
# MATCH DELIVERY TO SHIPMENT
# =========================================================

deliveries_checked = (
    bronze_deliveries.alias("d")
    .join(
        valid_shipment_ids.alias("s"),
        F.col("d.shipment_id")
        == F.col("s.valid_shipment_id"),
        "left"
    )
    .select(
        "d.*",
        F.col("s.valid_shipment_id")
    )
)

# =========================================================
# VALIDATE
# =========================================================

validated_deliveries = (
    deliveries_checked
    .withColumn(
        "rejection_reason",

        F.when(
            F.col("event_id").isNull(),
            F.lit("missing_delivery_event_id")
        )

        .when(
            F.col("shipment_id").isNull(),
            F.lit("missing_shipment_id")
        )

        .when(
            F.col("valid_shipment_id").isNull(),
            F.lit("unknown_shipment")
        )

        .when(
            F.col("event_time").isNull(),
            F.lit("invalid_event_time")
        )

        .when(
            F.col("status").isNull(),
            F.lit("missing_delivery_status")
        )

        .when(
            F.upper(F.col("status"))
            != F.lit("DELIVERED"),
            F.lit("invalid_delivery_status")
        )

        .otherwise(
            F.lit(None).cast("string")
        )
    )
)

# =========================================================
# DISPLAY
# =========================================================

display(
    validated_deliveries.select(
        "event_id",
        "shipment_id",
        "status",
        "event_time",
        "rejection_reason"
    )
)

In [0]:
from pyspark.sql import functions as F

# =========================================================
# SPLIT
# =========================================================

valid_deliveries = (
    validated_deliveries
    .filter(
        F.col("rejection_reason").isNull()
    )
)

quarantine_deliveries = (
    validated_deliveries
    .filter(
        F.col("rejection_reason").isNotNull()
    )
)

# =========================================================
# COUNTS
# =========================================================

bronze_delivery_count = bronze_deliveries.count()
valid_delivery_count = valid_deliveries.count()
quarantine_delivery_count = quarantine_deliveries.count()

print("Bronze deliveries     :", bronze_delivery_count)
print("Valid deliveries      :", valid_delivery_count)
print("Quarantine deliveries :", quarantine_delivery_count)

# =========================================================
# RECONCILIATION
# =========================================================

assert (
    valid_delivery_count
    + quarantine_delivery_count
    == bronze_delivery_count
), "FAILED: Delivery reconciliation failed"

print("✅ Delivery reconciliation passed")

In [0]:
from pyspark.sql import functions as F

# =========================================================
# CLEAN SILVER DELIVERY EVENTS
# =========================================================

silver_delivery_events = (
    valid_deliveries
    .select(
        "event_id",
        "shipment_id",
        F.upper(F.col("status")).alias("status"),
        "event_time",
        "_source_file",
        "_processing_time",
        "_run_id"
    )
)

# =========================================================
# WRITE SILVER
# =========================================================

(
    silver_delivery_events.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{NS}.silver_delivery_events"
    )
)

print("✅ silver_delivery_events written")

# =========================================================
# QUARANTINE
# =========================================================

quarantine_delivery_events = (
    quarantine_deliveries
    .select(
        "event_id",
        "shipment_id",
        "status",
        "event_time",
        "rejection_reason",
        "_source_file",
        "_processing_time",
        "_run_id"
    )
)

(
    quarantine_delivery_events.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{NS}.quarantine_delivery_events"
    )
)

print("✅ quarantine_delivery_events written")

# =========================================================
# DISPLAY
# =========================================================

print("")
print("SILVER DELIVERY EVENTS")

display(
    spark.table(
        f"{NS}.silver_delivery_events"
    )
)

In [0]:
from pyspark.sql import functions as F

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"
NS = f"{CATALOG}.{SCHEMA}"

silver_deliveries = spark.table(
    f"{NS}.silver_delivery_events"
)

sh003_delivery_count = (
    silver_deliveries
    .filter(
        F.col("shipment_id") == "SH003"
    )
    .count()
)

print(
    "Delivery events for SH003:",
    sh003_delivery_count
)

assert sh003_delivery_count == 0, (
    f"FAILED: SH003 should have no delivery event, "
    f"but found {sh003_delivery_count}"
)

print(
    "✅ SH003 correctly remains undelivered"
)

In [0]:
from pyspark.sql import functions as F

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"

NS = f"{CATALOG}.{SCHEMA}"

# =========================================================
# LOAD SILVER TABLES
# =========================================================

sensors = spark.table(
    f"{NS}.silver_sensor_events"
)

shipments = spark.table(
    f"{NS}.silver_shipments"
)

# =========================================================
# SELECT ONLY REQUIRED SHIPMENT FIELDS
# =========================================================

shipment_lookup = (
    shipments
    .select(
        "shipment_id",
        "product_id",
        "vehicle_id",
        "origin",
        "destination",
        "dispatch_time",
        "expected_arrival"
    )
)

# =========================================================
# JOIN
# =========================================================

sensor_with_shipment = (
    sensors.alias("sensor")
    .join(
        shipment_lookup.alias("shipment"),
        F.col("sensor.shipment_id")
        == F.col("shipment.shipment_id"),
        "inner"
    )
    .select(
        F.col("sensor.event_id"),
        F.col("sensor.shipment_id"),
        F.col("sensor.sensor_id"),
        F.col("sensor.event_time"),
        F.col("sensor.temperature_c"),

        F.col("shipment.product_id"),
        F.col("shipment.vehicle_id"),
        F.col("shipment.origin"),
        F.col("shipment.destination"),
        F.col("shipment.dispatch_time"),
        F.col("shipment.expected_arrival"),

        F.col("sensor._source_file"),
        F.col("sensor._processing_time"),
        F.col("sensor._run_id")
    )
)

print(
    "Sensor Silver count:",
    sensors.count()
)

print(
    "After shipment join:",
    sensor_with_shipment.count()
)

display(sensor_with_shipment)

In [0]:
from pyspark.sql import functions as F

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"

NS = f"{CATALOG}.{SCHEMA}"

# =========================================================
# LOAD SILVER TABLES
# =========================================================

sensors = spark.table(
    f"{NS}.silver_sensor_events"
)

shipments = spark.table(
    f"{NS}.silver_shipments"
)

# =========================================================
# SELECT ONLY REQUIRED SHIPMENT FIELDS
# =========================================================

shipment_lookup = (
    shipments
    .select(
        "shipment_id",
        "product_id",
        "vehicle_id",
        "origin",
        "destination",
        "dispatch_time",
        "expected_arrival"
    )
)

# =========================================================
# JOIN
# =========================================================

sensor_with_shipment = (
    sensors.alias("sensor")
    .join(
        shipment_lookup.alias("shipment"),
        F.col("sensor.shipment_id")
        == F.col("shipment.shipment_id"),
        "inner"
    )
    .select(
        F.col("sensor.event_id"),
        F.col("sensor.shipment_id"),
        F.col("sensor.sensor_id"),
        F.col("sensor.event_time"),
        F.col("sensor.temperature_c"),

        F.col("shipment.product_id"),
        F.col("shipment.vehicle_id"),
        F.col("shipment.origin"),
        F.col("shipment.destination"),
        F.col("shipment.dispatch_time"),
        F.col("shipment.expected_arrival"),

        F.col("sensor._source_file"),
        F.col("sensor._processing_time"),
        F.col("sensor._run_id")
    )
)

print(
    "Sensor Silver count:",
    sensors.count()
)

print(
    "After shipment join:",
    sensor_with_shipment.count()
)

display(sensor_with_shipment)

In [0]:
assert (
    sensor_with_shipment.count()
    == sensors.count()
), (
    "FAILED: Sensor records were lost "
    "during shipment enrichment"
)

print("✅")

In [0]:
from pyspark.sql import functions as F

# =========================================================
# STEPS 58-64: COMPLETE SENSOR ENRICHMENT
# =========================================================

# ---------------------------------------------------------
# 1. CONFIGURATION
# ---------------------------------------------------------

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"

NS = f"{CATALOG}.{SCHEMA}"

spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

print("Namespace:", NS)


# ---------------------------------------------------------
# 2. LOAD SAVED SILVER TABLES
# ---------------------------------------------------------

sensors = spark.table(
    f"{NS}.silver_sensor_events"
)

shipments = spark.table(
    f"{NS}.silver_shipments"
)

products = spark.table(
    f"{NS}.silver_product_limits"
)


# ---------------------------------------------------------
# 3. DISPLAY INPUT COUNTS
# ---------------------------------------------------------

sensor_count = sensors.count()
shipment_count = shipments.count()
product_count = products.count()

print("")
print("==========================================")
print("SILVER ENRICHMENT INPUT COUNTS")
print("==========================================")
print("Silver sensors   :", sensor_count)
print("Silver shipments :", shipment_count)
print("Silver products  :", product_count)


# =========================================================
# STEP 58: JOIN SENSOR EVENTS WITH SHIPMENTS
# =========================================================

shipment_lookup = (
    shipments
    .select(
        "shipment_id",
        "product_id",
        "vehicle_id",
        "origin",
        "destination",
        "dispatch_time",
        "expected_arrival"
    )
)

sensor_with_shipment = (
    sensors.alias("sensor")
    .join(
        shipment_lookup.alias("shipment"),
        F.col("sensor.shipment_id")
        == F.col("shipment.shipment_id"),
        "inner"
    )
    .select(
        F.col("sensor.event_id").alias("event_id"),
        F.col("sensor.shipment_id").alias("shipment_id"),
        F.col("sensor.sensor_id").alias("sensor_id"),
        F.col("sensor.event_time").alias("event_time"),
        F.col("sensor.temperature_c").alias("temperature_c"),

        F.col("shipment.product_id").alias("product_id"),
        F.col("shipment.vehicle_id").alias("vehicle_id"),
        F.col("shipment.origin").alias("origin"),
        F.col("shipment.destination").alias("destination"),
        F.col("shipment.dispatch_time").alias("dispatch_time"),
        F.col("shipment.expected_arrival").alias(
            "expected_arrival"
        ),

        F.col("sensor._source_file").alias("_source_file"),
        F.col("sensor._processing_time").alias(
            "_processing_time"
        ),
        F.col("sensor._run_id").alias("_run_id")
    )
)

sensor_with_shipment_count = (
    sensor_with_shipment.count()
)

print("")
print("==========================================")
print("SENSOR TO SHIPMENT JOIN")
print("==========================================")
print("Sensor records before join :", sensor_count)
print(
    "Sensor records after join  :",
    sensor_with_shipment_count
)

assert sensor_with_shipment_count == sensor_count, (
    "FAILED: Sensor records were lost during "
    "the Sensor-to-Shipment join. "
    f"Before={sensor_count}, "
    f"After={sensor_with_shipment_count}"
)

print("✅ Sensor-to-Shipment join passed")


# =========================================================
# STEP 59: JOIN PRODUCT TEMPERATURE LIMITS
# =========================================================

product_lookup = (
    products
    .select(
        "product_id",
        "min_temperature",
        "max_temperature"
    )
)

sensor_enriched_base = (
    sensor_with_shipment.alias("sensor")
    .join(
        product_lookup.alias("product"),
        F.col("sensor.product_id")
        == F.col("product.product_id"),
        "inner"
    )
    .select(
        F.col("sensor.event_id").alias("event_id"),
        F.col("sensor.shipment_id").alias("shipment_id"),
        F.col("sensor.sensor_id").alias("sensor_id"),
        F.col("sensor.event_time").alias("event_time"),
        F.col("sensor.temperature_c").alias(
            "temperature_c"
        ),

        F.col("sensor.product_id").alias("product_id"),
        F.col("product.min_temperature").alias(
            "min_temperature"
        ),
        F.col("product.max_temperature").alias(
            "max_temperature"
        ),

        F.col("sensor.vehicle_id").alias("vehicle_id"),
        F.col("sensor.origin").alias("origin"),
        F.col("sensor.destination").alias("destination"),
        F.col("sensor.dispatch_time").alias(
            "dispatch_time"
        ),
        F.col("sensor.expected_arrival").alias(
            "expected_arrival"
        ),

        F.col("sensor._source_file").alias("_source_file"),
        F.col("sensor._processing_time").alias(
            "_processing_time"
        ),
        F.col("sensor._run_id").alias("_run_id")
    )
)

sensor_enriched_base_count = (
    sensor_enriched_base.count()
)

print("")
print("==========================================")
print("SENSOR TO PRODUCT JOIN")
print("==========================================")
print(
    "Records before product join :",
    sensor_with_shipment_count
)
print(
    "Records after product join  :",
    sensor_enriched_base_count
)

assert (
    sensor_enriched_base_count
    == sensor_with_shipment_count
), (
    "FAILED: Sensor records were lost during "
    "the Product Limits join. "
    f"Before={sensor_with_shipment_count}, "
    f"After={sensor_enriched_base_count}"
)

print("✅ Sensor-to-Product join passed")


# =========================================================
# STEP 60: CREATE TEMPERATURE EXCEPTION FLAG
# =========================================================

sensor_enriched = (
    sensor_enriched_base
    .withColumn(
        "is_temperature_exception",
        (
            F.col("temperature_c")
            < F.col("min_temperature")
        )
        |
        (
            F.col("temperature_c")
            > F.col("max_temperature")
        )
    )
)

print("")
print("==========================================")
print("TEMPERATURE CLASSIFICATION")
print("==========================================")

display(
    sensor_enriched
    .select(
        "event_id",
        "shipment_id",
        "product_id",
        "temperature_c",
        "min_temperature",
        "max_temperature",
        "is_temperature_exception"
    )
    .orderBy(
        "event_time",
        "event_id"
    )
)


# =========================================================
# STEP 61: VERIFY E004
# =========================================================

e004 = (
    sensor_enriched
    .filter(
        F.col("event_id") == "E004"
    )
)

print

In [0]:
from pyspark.sql import functions as F

# =========================================================
# STEP 65: FINAL FULL SILVER CERTIFICATE
# =========================================================

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"
NS = f"{CATALOG}.{SCHEMA}"

# =========================================================
# LOAD SAVED SILVER AND QUARANTINE TABLES
# =========================================================

products = spark.table(
    f"{NS}.silver_product_limits"
)

shipments = spark.table(
    f"{NS}.silver_shipments"
)

deliveries = spark.table(
    f"{NS}.silver_delivery_events"
)

sensors = spark.table(
    f"{NS}.silver_sensor_events"
)

enriched = spark.table(
    f"{NS}.silver_sensor_enriched"
)

sensor_quarantine = spark.table(
    f"{NS}.quarantine_sensor_events"
)

# =========================================================
# TABLE COUNTS
# =========================================================

product_count = products.count()
shipment_count = shipments.count()
delivery_count = deliveries.count()
sensor_count = sensors.count()
enriched_count = enriched.count()
sensor_quarantine_count = sensor_quarantine.count()

# =========================================================
# TEMPERATURE EXCEPTION COUNTS
# =========================================================

temperature_exception_count = (
    enriched
    .filter(
        F.col("is_temperature_exception") == True
    )
    .count()
)

# =========================================================
# EXPECTED EXCEPTION EVENTS
# =========================================================

e004_exception_count = (
    enriched
    .filter(
        (F.col("event_id") == "E004")
        &
        (F.col("is_temperature_exception") == True)
    )
    .count()
)

e008_exception_count = (
    enriched
    .filter(
        (F.col("event_id") == "E008")
        &
        (F.col("is_temperature_exception") == True)
    )
    .count()
)

e011_exception_count = (
    enriched
    .filter(
        (F.col("event_id") == "E011")
        &
        (F.col("is_temperature_exception") == True)
    )
    .count()
)

# =========================================================
# BOUNDARY VALUE TESTS
# =========================================================

e003_incorrect_exception_count = (
    enriched
    .filter(
        (F.col("event_id") == "E003")
        &
        (F.col("is_temperature_exception") == True)
    )
    .count()
)

e007_incorrect_exception_count = (
    enriched
    .filter(
        (F.col("event_id") == "E007")
        &
        (F.col("is_temperature_exception") == True)
    )
    .count()
)

# =========================================================
# SILVER SENSOR QUALITY TESTS
# =========================================================

null_event_ids_in_silver = (
    sensors
    .filter(
        F.col("event_id").isNull()
    )
    .count()
)

unknown_shipments_in_silver = (
    sensors
    .filter(
        F.col("shipment_id") == "SH999"
    )
    .count()
)

duplicate_event_ids_in_silver = (
    sensors
    .groupBy("event_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

# =========================================================
# QUARANTINE TESTS
# =========================================================

hot_quarantine_count = (
    sensor_quarantine
    .filter(
        F.col("temperature_c") == "hot"
    )
    .count()
)

sh999_quarantine_count = (
    sensor_quarantine
    .filter(
        F.col("shipment_id") == "SH999"
    )
    .count()
)

missing_event_id_quarantine_count = (
    sensor_quarantine
    .filter(
        F.col("event_id").isNull()
    )
    .count()
)

# =========================================================
# DUPLICATE E015 TEST
# =========================================================

e015_silver_count = (
    sensors
    .filter(
        F.col("event_id") == "E015"
    )
    .count()
)

# =========================================================
# SH003 MUST REMAIN UNDELIVERED
# =========================================================

sh003_delivery_count = (
    deliveries
    .filter(
        F.col("shipment_id") == "SH003"
    )
    .count()
)

# =========================================================
# CHECK FOR BROKEN ENRICHMENT JOINS
# =========================================================

missing_product_limits_count = (
    enriched
    .filter(
        F.col("product_id").isNull()
        |
        F.col("min_temperature").isNull()
        |
        F.col("max_temperature").isNull()
    )
    .count()
)

# =========================================================
# PRINT THE FINAL REPORT
# =========================================================

print("")
print("================================================")
print("           FINAL FULL SILVER REPORT")
print("================================================")

print("")
print("REFERENCE SILVER TABLES")
print("------------------------------------------------")
print("Silver products                  :", product_count)
print("Silver shipments                 :", shipment_count)
print("Silver deliveries                :", delivery_count)

print("")
print("SENSOR SILVER TABLES")
print("------------------------------------------------")
print("Silver sensor events             :", sensor_count)
print("Enriched sensor events           :", enriched_count)
print("Quarantined sensor events        :", sensor_quarantine_count)

print("")
print("TEMPERATURE EXCEPTIONS")
print("------------------------------------------------")
print("Total temperature exceptions     :", temperature_exception_count)
print("E004 correctly flagged           :", e004_exception_count)
print("E008 correctly flagged           :", e008_exception_count)
print("E011 correctly flagged           :", e011_exception_count)

print("")
print("BOUNDARY VALIDATION")
print("------------------------------------------------")
print("E003 incorrectly flagged         :", e003_incorrect_exception_count)
print("E007 incorrectly flagged         :", e007_incorrect_exception_count)

print("")
print("SENSOR QUALITY")
print("------------------------------------------------")
print("NULL event IDs in Silver         :", null_event_ids_in_silver)
print("SH999 records in Silver          :", unknown_shipments_in_silver)
print("Duplicate IDs in Silver          :", duplicate_event_ids_in_silver)
print("E015 occurrences in Silver       :", e015_silver_count)

print("")
print("QUARANTINE VALIDATION")
print("------------------------------------------------")
print("'hot' records quarantined        :", hot_quarantine_count)
print("SH999 records quarantined        :", sh999_quarantine_count)
print("NULL event IDs quarantined       :", missing_event_id_quarantine_count)

print("")
print("ENRICHMENT VALIDATION")
print("------------------------------------------------")
print("Missing product limits           :", missing_product_limits_count)
print("SH003 delivery events            :", sh003_delivery_count)

print("")
print("================================================")

# =========================================================
# ASSERT EXPECTED FIXTURE RESULTS
# =========================================================

assert product_count == 3, (
    f"FAILED: Expected 3 Silver products, "
    f"found {product_count}"
)

assert shipment_count == 5, (
    f"FAILED: Expected 5 Silver shipments, "
    f"found {shipment_count}"
)

assert delivery_count == 4, (
    f"FAILED: Expected 4 Silver deliveries, "
    f"found {delivery_count}"
)

# From 20 Bronze sensor records:
# 3 invalid records go to quarantine.
# 1 repeated E015 record is removed during deduplication.
# Therefore Silver should contain 16 records.

assert sensor_count == 16

In [0]:
from pyspark.sql import functions as F

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"
NS = f"{CATALOG}.{SCHEMA}"

# Load existing Silver tables
sensors = spark.table(
    f"{NS}.silver_sensor_events"
)

shipments = spark.table(
    f"{NS}.silver_shipments"
)

products = spark.table(
    f"{NS}.silver_product_limits"
)

# Sensor -> Shipment
sensor_shipment = (
    sensors.alias("s")
    .join(
        shipments.alias("sh"),
        F.col("s.shipment_id") == F.col("sh.shipment_id"),
        "inner"
    )
    .select(
        F.col("s.event_id"),
        F.col("s.shipment_id"),
        F.col("s.sensor_id"),
        F.col("s.event_time"),
        F.col("s.temperature_c"),

        F.col("sh.product_id"),
        F.col("sh.vehicle_id"),
        F.col("sh.origin"),
        F.col("sh.destination"),
        F.col("sh.dispatch_time"),
        F.col("sh.expected_arrival"),

        F.col("s._source_file"),
        F.col("s._processing_time"),
        F.col("s._run_id")
    )
)

# Shipment/Product -> Product limits
enriched = (
    sensor_shipment.alias("s")
    .join(
        products.alias("p"),
        F.col("s.product_id") == F.col("p.product_id"),
        "inner"
    )
    .select(
        F.col("s.event_id"),
        F.col("s.shipment_id"),
        F.col("s.sensor_id"),
        F.col("s.event_time"),
        F.col("s.temperature_c"),

        F.col("s.product_id"),
        F.col("p.min_temperature"),
        F.col("p.max_temperature"),

        F.col("s.vehicle_id"),
        F.col("s.origin"),
        F.col("s.destination"),
        F.col("s.dispatch_time"),
        F.col("s.expected_arrival"),

        F.col("s._source_file"),
        F.col("s._processing_time"),
        F.col("s._run_id")
    )
)

# Business rule:
# below minimum OR above maximum = temperature exception
enriched = (
    enriched
    .withColumn(
        "is_temperature_exception",
        (
            F.col("temperature_c") < F.col("min_temperature")
        )
        |
        (
            F.col("temperature_c") > F.col("max_temperature")
        )
    )
)

# Verify no records were lost
sensor_count = sensors.count()
enriched_count = enriched.count()

print("Silver sensors :", sensor_count)
print("Enriched rows  :", enriched_count)

assert sensor_count == enriched_count, (
    f"FAILED: Join lost records. "
    f"Sensor={sensor_count}, Enriched={enriched_count}"
)

# Persist the table
(
    enriched.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{NS}.silver_sensor_enriched"
    )
)

print("")
print("✅ silver_sensor_enriched CREATED")

# Read it back from Unity Catalog
saved_table = spark.table(
    f"{NS}.silver_sensor_enriched"
)

print(
    "Persisted row count:",
    saved_table.count()
)

display(
    saved_table
    .select(
        "event_id",
        "shipment_id",
        "product_id",
        "temperature_c",
        "min_temperature",
        "max_temperature",
        "is_temperature_exception"
    )
    .orderBy("event_time")
)

print("")
print("✅ silver_sensor_enriched VERIFIED")

In [0]:
%sql

SHOW TABLES IN training_lab_classic.cold_chain
LIKE 'silver_sensor_enriched';

In [0]:
%sql

SELECT COUNT(*) AS total_records
FROM training_lab_classic.cold_chain.silver_sensor_enriched;

In [0]:
%sql

SELECT
    event_id,
    shipment_id,
    product_id,
    temperature_c,
    min_temperature,
    max_temperature,
    is_temperature_exception
FROM training_lab_classic.cold_chain.silver_sensor_enriched
WHERE is_temperature_exception = true
ORDER BY event_id;

In [0]:
%sql

SELECT
    event_id,
    product_id,
    temperature_c,
    min_temperature,
    max_temperature,
    is_temperature_exception
FROM training_lab_classic.cold_chain.silver_sensor_enriched
WHERE event_id IN ('E003', 'E007')
ORDER BY event_id;

In [0]:
from pyspark.sql import functions as F

# =========================================================
# STEP 65: FINAL FULL SILVER CERTIFICATE
# =========================================================

# ---------------------------------------------------------
# 1. CONFIGURATION
# ---------------------------------------------------------

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"

NS = f"{CATALOG}.{SCHEMA}"

print("Namespace:", NS)


# ---------------------------------------------------------
# 2. LOAD ALL PERSISTED SILVER TABLES
# ---------------------------------------------------------

products = spark.table(
    f"{NS}.silver_product_limits"
)

shipments = spark.table(
    f"{NS}.silver_shipments"
)

deliveries = spark.table(
    f"{NS}.silver_delivery_events"
)

sensors = spark.table(
    f"{NS}.silver_sensor_events"
)

enriched = spark.table(
    f"{NS}.silver_sensor_enriched"
)

sensor_quarantine = spark.table(
    f"{NS}.quarantine_sensor_events"
)


# ---------------------------------------------------------
# 3. BASIC TABLE COUNTS
# ---------------------------------------------------------

product_count = products.count()

shipment_count = shipments.count()

delivery_count = deliveries.count()

sensor_count = sensors.count()

enriched_count = enriched.count()

sensor_quarantine_count = sensor_quarantine.count()


# ---------------------------------------------------------
# 4. TOTAL TEMPERATURE EXCEPTIONS
# ---------------------------------------------------------

temperature_exception_count = (
    enriched
    .filter(
        F.col("is_temperature_exception") == True
    )
    .count()
)


# ---------------------------------------------------------
# 5. VERIFY E004
# SH001 -> P001
# Temperature = 11
# Allowed = 2 to 8
# Must be exception
# ---------------------------------------------------------

e004_exception_count = (
    enriched
    .filter(
        (F.col("event_id") == "E004")
        &
        (F.col("is_temperature_exception") == True)
    )
    .count()
)


# ---------------------------------------------------------
# 6. VERIFY E008
# SH002 -> P002
# Temperature = 9
# Allowed = 3 to 7
# Must be exception
# ---------------------------------------------------------

e008_exception_count = (
    enriched
    .filter(
        (F.col("event_id") == "E008")
        &
        (F.col("is_temperature_exception") == True)
    )
    .count()
)


# ---------------------------------------------------------
# 7. VERIFY E011
# SH003 -> P003
# Temperature = -1
# Allowed = 1 to 5
# Must be exception
# ---------------------------------------------------------

e011_exception_count = (
    enriched
    .filter(
        (F.col("event_id") == "E011")
        &
        (F.col("is_temperature_exception") == True)
    )
    .count()
)


# ---------------------------------------------------------
# 8. VERIFY E003 BOUNDARY
# P001 maximum = 8
# E003 temperature = 8
# Must NOT be exception
# ---------------------------------------------------------

e003_incorrect_exception_count = (
    enriched
    .filter(
        (F.col("event_id") == "E003")
        &
        (F.col("is_temperature_exception") == True)
    )
    .count()
)


# ---------------------------------------------------------
# 9. VERIFY E007 BOUNDARY
# P002 minimum = 3
# E007 temperature = 3
# Must NOT be exception
# ---------------------------------------------------------

e007_incorrect_exception_count = (
    enriched
    .filter(
        (F.col("event_id") == "E007")
        &
        (F.col("is_temperature_exception") == True)
    )
    .count()
)


# ---------------------------------------------------------
# 10. VERIFY SILVER HAS NO NULL EVENT IDs
# ---------------------------------------------------------

null_event_ids_in_silver = (
    sensors
    .filter(
        F.col("event_id").isNull()
    )
    .count()
)


# ---------------------------------------------------------
# 11. VERIFY SH999 IS NOT IN SILVER
# ---------------------------------------------------------

unknown_shipments_in_silver = (
    sensors
    .filter(
        F.col("shipment_id") == "SH999"
    )
    .count()
)


# ---------------------------------------------------------
# 12. VERIFY NO DUPLICATE EVENT IDS REMAIN
# ---------------------------------------------------------

duplicate_event_ids_in_silver = (
    sensors
    .groupBy("event_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


# ---------------------------------------------------------
# 13. VERIFY E015 DEDUPLICATION
# ---------------------------------------------------------

e015_silver_count = (
    sensors
    .filter(
        F.col("event_id") == "E015"
    )
    .count()
)


# ---------------------------------------------------------
# 14. VERIFY "hot" IS IN QUARANTINE
# ---------------------------------------------------------

hot_quarantine_count = (
    sensor_quarantine
    .filter(
        F.col("temperature_c") == "hot"
    )
    .count()
)


# ---------------------------------------------------------
# 15. VERIFY SH999 IS IN QUARANTINE
# ---------------------------------------------------------

sh999_quarantine_count = (
    sensor_quarantine
    .filter(
        F.col("shipment_id") == "SH999"
    )
    .count()
)


# ---------------------------------------------------------
# 16. VERIFY NULL EVENT ID IS IN QUARANTINE
# ---------------------------------------------------------

missing_event_id_quarantine_count = (
    sensor_quarantine
    .filter(
        F.col("event_id").isNull()
    )
    .count()
)


# ---------------------------------------------------------
# 17. VERIFY NO PRODUCT INFORMATION WAS LOST
# DURING ENRICHMENT
# ---------------------------------------------------------

missing_product_limits_count = (
    enriched
    .filter(
        F.col("product_id").isNull()
        |
        F.col("min_temperature").isNull()
        |
        F.col("max_temperature").isNull()
    )
    .count()
)


# ---------------------------------------------------------
# 18. VERIFY SH003 HAS NO DELIVERY
# ---------------------------------------------------------

sh003_delivery_count = (
    deliveries
    .filter(
        F.col("shipment_id") == "SH003"
    )
    .count()
)


# ---------------------------------------------------------
# 19. VERIFY E004 IS STILL PRESENT IN ENRICHED SILVER
# ---------------------------------------------------------

e004_enriched_count = (
    enriched
    .filter(
        F.col("event_id") == "E004"
    )
    .count()
)


# ---------------------------------------------------------
# 20. VERIFY SILVER SENSOR SCHEMA
# ---------------------------------------------------------

sensor_schema = {
    field.name: field.dataType.simpleString()
    for field in sensors.schema.fields
}

temperature_type = sensor_schema.get(
    "temperature_c"
)

event_time_type = sensor_schema.get(
    "event_time"
)


# =========================================================
# 21. PRINT REPORT
# =========================================================

print("")
print("================================================")
print("           FINAL FULL SILVER REPORT")
print("================================================")

print("")
print("REFERENCE SILVER TABLES")
print("------------------------------------------------")
print("Silver products                  :", product_count)
print("Silver shipments                 :", shipment_count)
print("Silver deliveries                :", delivery_count)

print("")
print("SENSOR TABLES")
print("------------------------------------------------")
print("Silver sensor events             :", sensor_count)
print("Enriched sensor events           :", enriched_count)
print("Quarantined sensor events        :", sensor_quarantine_count)

print("")
print("TEMPERATURE EXCEPTIONS")
print("------------------------------------------------")
print("Total exceptions                 :", temperature_exception_count)
print("E004 correctly flagged           :", e004_exception_count)
print("E008 correctly flagged           :", e008_exception_count)
print("E011 correctly flagged           :", e011_exception_count)